In [1]:
import pandas as pd
import numpy as np

from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import MinMaxScaler

In [2]:
def load_data(path):
    return pd.read_csv(path)

df = load_data("trade_1988_2021.csv")
df.head()

,ReporterISO3,ReporterName,PartnerISO3,PartnerName,Year,TradeFlowName,TradeValue in 1000 USD
0,AFG,Afghanistan,SWE,Sweden,2017,Export,86.752
1,AFG,Afghanistan,JOR,Jordan,2018,Export,2796.481
2,AFG,Afghanistan,JOR,Jordan,2017,Export,3100.187
3,AFG,Afghanistan,ITA,Italy,2018,Export,279.918
4,AFG,Afghanistan,ITA,Italy,2017,Export,416.642


In [3]:
#исправление дубликатов
def clean_data(df):
    df = df.drop_duplicates()
    df = df[df["TradeValue in 1000 USD"] >= 0]
    return df

df = clean_data(df)

In [4]:
categorical_cols = df.select_dtypes(include="object").columns
categorical_cols

Index(['ReporterISO3', 'ReporterName', 'PartnerISO3', 'PartnerName',
       'TradeFlowName'],
      dtype='object')

In [5]:
for col in categorical_cols:
    print(col, df[col].nunique())

ReporterISO3 207
ReporterName 207
PartnerISO3 258
PartnerName 258
TradeFlowName 1


In [6]:
def label_encode_column(df, column):
    le = LabelEncoder()
    df[column + "_LE"] = le.fit_transform(df[column])
    return df

df = label_encode_column(df, "ReporterName")

In [7]:
def onehot_encode_column(df, column):
    encoder = OneHotEncoder(sparse_output=False, drop="first")
    encoded = encoder.fit_transform(df[[column]])
    encoded_df = pd.DataFrame(encoded, 
                              columns=encoder.get_feature_names_out([column]))
    df = df.reset_index(drop=True)
    encoded_df = encoded_df.reset_index(drop=True)
    return pd.concat([df, encoded_df], axis=1)

df = onehot_encode_column(df, "Year")

In [8]:
numeric_cols = df.select_dtypes(include=["int64", "float64"]).columns
numeric_cols

Index(['Year', 'TradeValue in 1000 USD', 'ReporterName_LE', 'Year_1989',
       'Year_1990', 'Year_1991', 'Year_1992', 'Year_1993', 'Year_1994',
       'Year_1995', 'Year_1996', 'Year_1997', 'Year_1998', 'Year_1999',
       'Year_2000', 'Year_2001', 'Year_2002', 'Year_2003', 'Year_2004',
       'Year_2005', 'Year_2006', 'Year_2007', 'Year_2008', 'Year_2009',
       'Year_2010', 'Year_2011', 'Year_2012', 'Year_2013', 'Year_2014',
       'Year_2015', 'Year_2016', 'Year_2017', 'Year_2018', 'Year_2019',
       'Year_2020', 'Year_2021'],
      dtype='object')

In [9]:
def standard_scale(df, column):
    scaler = StandardScaler()
    df[column + "_scaled"] = scaler.fit_transform(df[[column]])
    return df

df = standard_scale(df, "TradeValue in 1000 USD")

In [10]:
df["Log_TradeValue"] = np.log1p(df["TradeValue in 1000 USD"])

In [11]:
df["Decade"] = (df["Year"] // 10) * 10

In [12]:
def drop_irrelevant(df):
    columns_to_drop = [
        "ReporterName",
        "PartnerName"
    ]
    return df.drop(columns=columns_to_drop)

df_final = drop_irrelevant(df)

Текстовые признаки ReporterName и PartnerName были удалены, поскольку алгоритмы машинного обучения не работают с текстовыми значениями напрямую. Вместо них использованы закодированные числовые представления.

In [13]:
df_final.shape
df_final.head()

,ReporterISO3,PartnerISO3,Year,TradeFlowName,TradeValue in 1000 USD,ReporterName_LE,Year_1989,Year_1990,Year_1991,Year_1992,...,Year_2015,Year_2016,Year_2017,Year_2018,Year_2019,Year_2020,Year_2021,TradeValue in 1000 USD_scaled,Log_TradeValue,Decade
0,AFG,SWE,2017,Export,86.752,0,0.0,0.0,0.0,0.0,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,-0.054708,4.474515,2010
1,AFG,JOR,2018,Export,2796.481,0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,-0.054582,7.936475,2010
2,AFG,JOR,2017,Export,3100.187,0,0.0,0.0,0.0,0.0,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,-0.054568,8.039540,2010
3,AFG,ITA,2018,Export,279.918,0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,-0.054699,5.638063,2010
4,AFG,ITA,2017,Export,416.642,0,0.0,0.0,0.0,0.0,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,-0.054693,6.034625,2010


Вывод
В ходе лабораторной работы были выполнены:
-очистка данных
-анализ категориальных признаков
-кодирование признаков двумя методами (Label Encoding и One-Hot Encoding)
-масштабирование числовых признаков (StandardScaler и MinMaxScaler)
-создание новых признаков
-удаление нерелевантных признаков
Данные подготовлены для дальнейшего применения алгоритмов машинного обучения.